# Feature reduction without Stockfish

Which columns can go? First the false predictors (known only after the move or the game), then the inspection of what is left: missing values, near-constant columns, how much every column tells about blunders, correlations. Only training games are looked at. Run `inspection.py` first.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "config.yaml").exists():
    root = root.parent
sys.path.insert(0, str(root))

import importlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from src.common.plot_style import INK, MUTED, WIDTH, save
from src.config import CFG

paths = importlib.import_module("src.02_preprocessing.common.paths")
false_predictors = importlib.import_module("src.02_preprocessing.common.false_predictors")
FOLDER = paths.WITHOUT_DIR
INSPECTION = FOLDER / "inspection"
FIG_DIR = INSPECTION / "figures"
RULES = CFG["reduction"]

In [ ]:
columns = pd.read_csv(INSPECTION / "columns.csv")
corr = pd.read_csv(INSPECTION / "correlation.csv", index_col=0)
pairs = pd.read_csv(INSPECTION / "pairs.csv")
categories = pd.read_csv(INSPECTION / "categories.csv")
decisions = pd.read_csv(FOLDER / "final" / "removed.csv")
kept_anyway = pd.read_csv(FOLDER / "final" / "kept.csv")
removed = set(decisions["column"])
meta = json.loads((INSPECTION / "inspection.json").read_text(encoding="utf-8"))
decisions = pd.read_csv(FOLDER / "final" / "removed.csv")
kept_anyway = pd.read_csv(FOLDER / "final" / "kept.csv")
removed = set(decisions["column"])
meta

## 1. False predictors: removed before anything else

In [ ]:
rows = [{"column": c, "known": when, "reason": why} for c, (when, why) in false_predictors.FALSE_PREDICTORS.items()]
pd.DataFrame(rows)

## 2. The columns that are left

In [ ]:
columns.round(3)

## 3. Correlation between the columns

Spearman correlation (rank based, so curved relations count too). Dark red or dark blue off the diagonal means two columns carry almost the same information.

In [ ]:
size = max(6, 0.22 * len(corr))
fig, ax = plt.subplots(figsize=(size, size * 0.85))
im = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=6)
ax.set_yticks(range(len(corr)), corr.index, fontsize=6)
ax.grid(False)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    if label.get_text() in removed:
        label.set_color("#D55E00")
cb = fig.colorbar(im, ax=ax, fraction=.035, pad=.02)
cb.set_label("Spearman correlation")
ax.set_title("columns removed by the reduction in orange", fontsize=8, color=MUTED)
save(fig, FIG_DIR, "correlation_heatmap")
plt.show()

In [ ]:
pairs.round(3)

## 4. Relation to blunders

How much every column tells about blunders (mutual information; unlike a correlation it also sees relations that are not a straight line, and it works for text columns). The dashed line is the limit of the *weak* rule. Then the blunder rate per value of the category columns.

In [ ]:
d = columns.sort_values("mi_millibits")
weak = RULES["weak_share"] * d["mi_millibits"].max()
fig, ax = plt.subplots(figsize=(WIDTH, 0.16 * len(d) + 0.8))
colors = ["#D55E00" if c in removed else "#0072B2" for c in d["column"]]
ax.barh(d["column"], d["mi_millibits"].clip(lower=1e-3), color=colors)
ax.axvline(weak, color=INK, lw=.8, ls="--")
ax.set_xscale("log")
ax.set_xlabel("information about blunders (millibits, log scale)")
ax.tick_params(axis="y", labelsize=6)
key = [Patch(color="#0072B2", label="kept"), Patch(color="#D55E00", label="removed"),
       Line2D([], [], color=INK, lw=.8, ls="--", label=f"weak: below {100 * RULES['weak_share']:g}% of the best")]
ax.legend(handles=key, loc="lower right")
save(fig, FIG_DIR, "target_information")
plt.show()

In [ ]:
categories.round(2)

## 5. Decision: what the reduction removes and why

The rules of `common/reduction.py` (settings: `reduction` in `src/settings.py`): *domain*: the column follows from others, or a kept column covers it; *redundant*: of two columns with |Spearman| of at least 0.9, the one that tells less about blunders; *weak*: less than 1% of what the best column tells. Below the removed columns, then the columns kept whatever the numbers say, each with its reason.

In [ ]:
kept = [c for c in columns["column"] if c not in removed]
print(f"{len(columns)} columns after the false predictors, {len(decisions)} removed, {len(kept)} kept")
display(decisions)
kept_anyway